# Preparar os dados para a análise
**Fase 3 — Preparação dos Dados | CRISP-DM**

No `02` vimos como estão os dados e o que merece cuidado. Aqui vamos aplicar essas decisões e guardar as tabelas que vamos usar a seguir.

A ideia é simples: **o bruto fica como veio da fonte; o preparado fica separado, com os tratamentos explicados**. Não vamos corrigir ou apagar valores só para a tabela ficar mais bonita. Também não vamos inventar uma ligação entre unidades quando ainda não conseguimos confirmar essa ligação.

Este notebook lê a recolha guardada do indicador antigo do SNS. Não precisa de voltar à API e não depende de variáveis que tenham ficado abertas no `02`.

| Decisão | O que vamos fazer | Porquê |
|---|---|---|
| D01 — Período | Usar janeiro de 2024 a junho de 2026. | Evitar ligar os ACES anteriores às ULS e evitar a quebra de definição em julho de 2026. |
| D02 — Organização | Dar nomes claros às colunas, converter tipos e ordenar por unidade/mês. | Facilitar a análise e evitar contas sobre texto. |
| D03 — Qualidade | Verificar chaves, valores, totais e percentagens; parar se houver problemas. | Os problemas têm de ser investigados, não escondidos por uma correção automática. |
| D04 — Continuidade | Manter todas as linhas no recorte; usar apenas séries completas na evolução e previsão. | A correspondência dos dois nomes da Póvoa/Vila do Conde continua por confirmar. |
| D05 — Percentagens | Recalcular a partir das contagens e guardar também as percentagens publicadas. | Permitir agregação correta e verificar o arredondamento da fonte. |
| D06 — Geografia | Criar faixas comuns de inscritos a partir de janeiro de 2024 e manter os limites fixos. | Comparar regiões por dimensão sem definir grupos a partir da percentagem sem médico. |
| D07 — Previsão | Preparar pares de meses consecutivos, histórico e separação temporal. | Prever o mês seguinte com informação do mês de origem ou anterior. |
| D08 — Consultas | Preservar as taxas, mas não as usar como variáveis do modelo nesta versão. | A janela exata do indicador de um ano ainda não está confirmada. |

**Antes de executar:** colocar este ficheiro em `notebooks/`, com os três ficheiros da recolha do `02` em `data/input/`. Usar um kernel Python com `pandas`. Depois, executar as células por ordem, ou usar **Run All**.

## 1. Preparar as pastas e definir o âmbito

O notebook encontra a raiz pela presença de `notebooks/` e `data/`. Cria as pastas de resultados que faltarem. Os ficheiros brutos têm de existir: se faltarem, voltamos à recolha do `02`.

Ao repetir a execução, as tabelas e notas geradas por esta fase são atualizadas. Os ficheiros de `data/input/` são apenas lidos.

In [2]:
import hashlib
import json
from pathlib import Path

import pandas as pd
from IPython.display import display

atual = Path.cwd().resolve()
raiz = next(
    (
        p
        for p in [atual, *atual.parents]
        if (p / "notebooks").is_dir() and (p / "data").is_dir()
    ),
    None,
)
if raiz is None:
    raise FileNotFoundError("Abrir a pasta do projeto, com notebooks/ e data/.")

PASTA_INPUT = raiz / "data" / "input"
PASTA_OUTPUT = raiz / "data" / "output"
PASTA_ANALYTICS = raiz / "data" / "analytics"
PASTA_DOCS = raiz / "docs"
for pasta in [PASTA_OUTPUT, PASTA_ANALYTICS, PASTA_DOCS]:
    pasta.mkdir(parents=True, exist_ok=True)

INICIO = pd.Timestamp("2024-01-01")
FIM = pd.Timestamp("2026-06-01")
INICIO_TESTE = pd.Timestamp("2026-01-01")
MESES = pd.date_range(INICIO, FIM, freq="MS")
CHAVE = ["regiao_saude", "unidade_fonte", "mes"]


def sha256(ficheiro):
    return hashlib.sha256(ficheiro.read_bytes()).hexdigest()


def guardar_csv(tabela, nome, pasta=PASTA_OUTPUT):
    tabela.to_csv(
        pasta / nome, index=False, encoding="utf-8-sig", date_format="%Y-%m-%d"
    )


print("Período:", INICIO.strftime("%Y-%m"), "a", FIM.strftime("%Y-%m"))
print("Dados preparados: data/output/ | Verificações: data/analytics/")

Período: 2024-01 a 2026-06
Dados preparados: data/output/ | Verificações: data/analytics/


In [3]:
# teste de execução::

print("1 — Python está a responder", flush=True)

import hashlib
import json
from pathlib import Path

print("2 — A importar pandas", flush=True)
import pandas as pd

print("3 — A importar IPython", flush=True)
from IPython.display import display

print("4 — A obter a pasta atual", flush=True)
atual = Path.cwd().resolve()
print("Pasta atual:", atual, flush=True)

raiz = None
for p in [atual, *atual.parents]:
    print("A verificar:", p, flush=True)
    if (p / "notebooks").is_dir() and (p / "data").is_dir():
        raiz = p
        break

print("5 — Raiz encontrada:", raiz, flush=True)

if raiz is None:
    raise FileNotFoundError("Não encontrei notebooks/ e data/.")

for nome in ["data/output", "data/analytics", "docs"]:
    pasta = raiz / nome
    print("A preparar:", pasta, flush=True)
    pasta.mkdir(parents=True, exist_ok=True)

print("6 — Diagnóstico concluído", flush=True)

1 — Python está a responder
2 — A importar pandas
3 — A importar IPython
4 — A obter a pasta atual
Pasta atual: C:\Users\Tiago\Documents\trabalho2\notebooks
A verificar: C:\Users\Tiago\Documents\trabalho2\notebooks
A verificar: C:\Users\Tiago\Documents\trabalho2
5 — Raiz encontrada: C:\Users\Tiago\Documents\trabalho2
A preparar: C:\Users\Tiago\Documents\trabalho2\data\output
A preparar: C:\Users\Tiago\Documents\trabalho2\data\analytics
A preparar: C:\Users\Tiago\Documents\trabalho2\docs
6 — Diagnóstico concluído


## 2. Ler a recolha e confirmar que não foi alterada

O registo da recolha tem uma assinatura SHA-256 do CSV e dos metadados. Se os bytes tiverem mudado, a assinatura muda e o notebook para. Isto permite verificar que estamos a trabalhar sobre a mesma recolha.

A data de recolha é diferente do mês dos dados: o CSV foi recolhido em outubro de 2026, mas a série antiga termina em junho de 2026.

In [4]:
ficheiro_bruto = PASTA_INPUT / "utentes_csp_historico_bruto.csv"
ficheiro_api = PASTA_INPUT / "utentes_csp_metadados_api.json"
ficheiro_recolha = PASTA_INPUT / "utentes_csp_recolha.json"
ficheiros_input = [ficheiro_bruto, ficheiro_api, ficheiro_recolha]
faltam = [p.name for p in ficheiros_input if not p.is_file()]
if faltam:
    raise FileNotFoundError(
        f"Faltam ficheiros da recolha do 02 em data/input: {faltam}"
    )

recolha = json.loads(ficheiro_recolha.read_text(encoding="utf-8"))
if recolha.get("dataset") != "utentes-inscritos-em-cuidados-de-saude-primarios":
    raise ValueError("A recolha não corresponde ao indicador histórico escolhido.")
for ficheiro, campo in [
    (ficheiro_bruto, "sha256_csv"),
    (ficheiro_api, "sha256_metadados"),
]:
    if sha256(ficheiro) != recolha[campo]:
        raise ValueError(f"O ficheiro original foi alterado: {ficheiro.name}.")
assinaturas_antes = {p.name: sha256(p) for p in ficheiros_input}

bruto = pd.read_csv(ficheiro_bruto, sep=";", encoding="utf-8-sig", dtype="string")
print("Recolhido em UTC:", recolha["recolhido_em_utc"])
print("Histórico:", bruto.shape[0], "linhas e", bruto.shape[1], "colunas")
display(bruto[["periodo", "ars", "aces", "utentes_inscritos_csp"]].head())

Recolhido em UTC: 2026-10-13T17:20:14.911883+00:00
Histórico: 6450 linhas e 13 colunas


,periodo,ars,aces,utentes_inscritos_csp
0,2016-01,Alentejo,ACES Alentejo Central,166863
1,2016-01,Alentejo,ACES Alentejo Litoral,92693
2,2016-01,Alentejo,ACES Baixo Alentejo,125192
3,2016-01,Alentejo,ACES São Mamede,117937
4,2016-01,Algarve,ACES Algarve Barlavento,159567


## 3. Nomes das colunas e tipos de dados

O nome `aces` aparece na fonte mesmo depois de 2024, quando as designações já são de ULS. Vamos chamar-lhe `unidade_fonte` e manter o texto original, para não perder a ligação à recolha.

As contagens ficam como inteiros, os indicadores percentuais como números decimais e o mês como data. `2024-01-01` representa **janeiro de 2024**; o primeiro dia serve apenas para ordenar meses.

As coordenadas continuam como texto de referência. Não representam os limites da ULS e não são usadas para inferir território, população ou classificação urbana/rural.

In [5]:
NOMES = {
    "periodo": "periodo_fonte",
    "ars": "regiao_saude",
    "aces": "unidade_fonte",
    "localizacao_geografica": "coordenadas_fonte",
    "utentes_inscritos_csp": "inscritos",
    "total_utentes_com_mdf_atribuido": "com_medico",
    "total_utentes_com_mdf_atribuido0": "com_medico_pct_fonte",
    "total_utentes_sem_mdf_atribuido": "sem_medico",
    "total_utentes_sem_mdf_atribuido0": "sem_medico_pct_fonte",
    "total_utentes_sem_mdf_atribuido_por_opcao": "sem_medico_opcao",
    "total_utentes_sem_mdf_atribuido_por_opcao0": "sem_medico_opcao_pct_fonte",
    "taxa_de_utilizacao_consultas_medicas_1_ano_todos_os_utentes": "consultas_1ano_todos_pct_fonte",
    "taxa_de_utilizacao_consultas_medicas_1_ano_nos_utentes_sem_mdf": "consultas_1ano_sem_medico_pct_fonte",
}
faltam_colunas = sorted(set(NOMES) - set(bruto.columns))
if faltam_colunas:
    raise ValueError(f"O esquema mudou. Faltam colunas: {faltam_colunas}")
if (
    bruto.empty
    or bruto.isna().any().any()
    or bruto.apply(lambda x: x.str.strip().eq("")).any().any()
):
    raise ValueError(
        "O bruto está vazio ou tem valores em falta. Investigar antes de continuar."
    )

historico = bruto[list(NOMES)].rename(columns=NOMES).copy()
if not historico["periodo_fonte"].str.fullmatch(r"\d{4}-\d{2}").all():
    raise ValueError("Encontrado um período que não segue o formato AAAA-MM.")
historico["mes"] = pd.to_datetime(
    historico["periodo_fonte"], format="%Y-%m", errors="raise"
)
CONTAGENS = ["inscritos", "com_medico", "sem_medico", "sem_medico_opcao"]
PERCENTAGENS_FONTE = [c for c in historico if c.endswith("_pct_fonte")]
for coluna in CONTAGENS + PERCENTAGENS_FONTE:
    numeros = pd.to_numeric(historico[coluna], errors="raise")
    if coluna in CONTAGENS:
        if numeros.mod(1).ne(0).any():
            raise ValueError(f"Contagens com casas decimais em {coluna}.")
        historico[coluna] = numeros.astype("int64")
    else:
        historico[coluna] = numeros.astype("float64")

dicionario = pd.DataFrame(
    [
        {
            "campo_fonte": original,
            "campo_preparado": novo,
            "tipo": str(historico[novo].dtype),
        }
        for original, novo in NOMES.items()
    ]
)
guardar_csv(dicionario, "03_mapa_colunas.csv", PASTA_ANALYTICS)
display(dicionario)

,campo_fonte,campo_preparado,tipo
0,periodo,periodo_fonte,string
1,ars,regiao_saude,string
2,aces,unidade_fonte,string
3,localizacao_geografica,coordenadas_fonte,string
4,utentes_inscritos_csp,inscritos,int64
5,total_utentes_com_mdf_atribuido,com_medico,int64
6,total_utentes_com_mdf_atribuido0,com_medico_pct_fonte,float64
7,total_utentes_sem_mdf_atribuido,sem_medico,int64
8,total_utentes_sem_mdf_atribuido0,sem_medico_pct_fonte,float64
9,total_utentes_sem_mdf_atribuido_por_opcao,sem_medico_opcao,int64


## 4. Verificar a qualidade antes de selecionar as linhas

Voltamos a verificar os dados convertidos. Não vamos preencher valores em falta, eliminar duplicados ou limitar percentagens automaticamente. Se uma verificação falhar, guardamos o resultado e paramos para perceber o problema.

A categoria `sem_medico` fica separada de `sem_medico_opcao`, como na fonte. As três categorias devem somar o total de inscritos. Para as percentagens publicadas, aceitamos uma diferença até **0,01 pontos percentuais**, compatível com o arredondamento apresentado pela fonte.

In [6]:
RECALCULADAS = {
    "com_medico_pct": "com_medico",
    "sem_medico_pct": "sem_medico",
    "sem_medico_opcao_pct": "sem_medico_opcao",
}
for pct, contagem in RECALCULADAS.items():
    historico[pct] = 100 * historico[contagem] / historico["inscritos"]

diferencas = pd.DataFrame(
    {pct: (historico[pct] - historico[pct + "_fonte"]).abs() for pct in RECALCULADAS}
)
qualidade = pd.DataFrame(
    [
        [
            "Linhas com chave mensal duplicada",
            int(historico.duplicated(CHAVE, keep=False).sum()),
        ],
        [
            "Valores numéricos negativos",
            int(historico[CONTAGENS + PERCENTAGENS_FONTE].lt(0).sum().sum()),
        ],
        [
            "Percentagens publicadas fora de 0 a 100",
            int(
                (
                    historico[PERCENTAGENS_FONTE].lt(0)
                    | historico[PERCENTAGENS_FONTE].gt(100)
                )
                .sum()
                .sum()
            ),
        ],
        [
            "Inscritos iguais ou inferiores a zero",
            int(historico["inscritos"].le(0).sum()),
        ],
        [
            "Linhas em que as categorias não somam o total",
            int(historico[CONTAGENS[1:]].sum(axis=1).ne(historico["inscritos"]).sum()),
        ],
        [
            "Diferenças de percentagem superiores a 0,01 pp",
            int(diferencas.gt(0.01).sum().sum()),
        ],
    ],
    columns=["verificacao", "ocorrencias"],
)
guardar_csv(qualidade, "03_qualidade_historico.csv", PASTA_ANALYTICS)
display(qualidade)
print(
    "Maior diferença de arredondamento:", round(float(diferencas.max().max()), 6), "pp"
)
if qualidade["ocorrencias"].gt(0).any():
    raise ValueError(
        "Há problemas de qualidade. Consultar 03_qualidade_historico.csv antes de continuar."
    )

,verificacao,ocorrencias
0,Linhas com chave mensal duplicada,0
1,Valores numéricos negativos,0
2,Percentagens publicadas fora de 0 a 100,0
3,Inscritos iguais ou inferiores a zero,0
4,Linhas em que as categorias não somam o total,0
5,"Diferenças de percentagem superiores a 0,01 pp",0


Maior diferença de arredondamento: 0.005 pp


## 5. Aplicar o recorte e verificar a continuidade

Esta é a implementação da **D01**: filtrar os meses de janeiro de 2024 a junho de 2026, incluindo os dois limites. As linhas anteriores continuam no CSV original.

No recorte há 39 registos por mês, mas 40 designações ao longo do período. Duas são `CSP da ULS Póvoa Varzim / Vila Conde` e `CSP da ULS Póvoa Varzim / Vila do Conde`. A semelhança sugere uma alteração de escrita, mas isso não confirma que o território e o universo de inscritos sejam exatamente os mesmos.

Por isso, **não juntamos esses nomes nesta versão**. São mantidos nas fotografias mensais e nos totais regionais. Para comparar o início e o fim ou preparar previsões, usamos as 38 designações com os 30 meses. Esta restrição reduz a cobertura da análise temporal e fica identificada nos resultados.

Também não vamos dizer que um nome estável prova que o território nunca mudou: é um critério de continuidade dos registos, com essa limitação.

In [7]:
principal = historico.loc[historico["mes"].between(INICIO, FIM)].copy()
principal = principal.sort_values(CHAVE).reset_index(drop=True)
if principal.empty:
    raise ValueError("Não há dados no período definido.")
meses_observados = pd.DatetimeIndex(principal["mes"].unique()).sort_values()
if not meses_observados.equals(MESES):
    raise ValueError("Faltam meses no recorte. Não interpolar antes de investigar.")

cobertura = (
    principal.groupby("mes")
    .agg(linhas=("unidade_fonte", "size"), designacoes=("unidade_fonte", "nunique"))
    .reset_index()
)
if not cobertura["linhas"].eq(39).all() or not cobertura["designacoes"].eq(39).all():
    raise ValueError(
        "A cobertura mensal mudou face ao 02. Rever as decisões de continuidade."
    )
continuidade = (
    principal.groupby(["regiao_saude", "unidade_fonte"])
    .agg(
        primeiro_mes=("mes", "min"),
        ultimo_mes=("mes", "max"),
        meses_presentes=("mes", "nunique"),
    )
    .reset_index()
)
continuidade["serie_completa"] = continuidade["meses_presentes"].eq(len(MESES))
principal = principal.merge(
    continuidade[["regiao_saude", "unidade_fonte", "serie_completa"]],
    on=["regiao_saude", "unidade_fonte"],
    validate="many_to_one",
)
principal = principal.sort_values(CHAVE).reset_index(drop=True)
series = principal.loc[principal["serie_completa"]].copy().reset_index(drop=True)
fluxo = pd.DataFrame(
    [
        ["Histórico preservado no bruto", len(bruto)],
        ["Fora do período principal", len(historico) - len(principal)],
        ["Recorte principal — todos os registos", len(principal)],
        ["Dentro do recorte, fora das séries completas", len(principal) - len(series)],
        ["Séries completas para evolução/previsão", len(series)],
    ],
    columns=["etapa", "linhas"],
)
guardar_csv(fluxo, "03_fluxo_registos.csv", PASTA_ANALYTICS)
guardar_csv(continuidade, "03_continuidade_unidades.csv", PASTA_ANALYTICS)
guardar_csv(cobertura, "03_cobertura_mensal.csv", PASTA_ANALYTICS)
display(fluxo)
display(continuidade.loc[~continuidade["serie_completa"]])
print("Designações completas:", series["unidade_fonte"].nunique())

,etapa,linhas
0,Histórico preservado no bruto,6450
1,Fora do período principal,5280
2,Recorte principal — todos os registos,1170
3,"Dentro do recorte, fora das séries completas",30
4,Séries completas para evolução/previsão,1140


,regiao_saude,unidade_fonte,primeiro_mes,ultimo_mes,meses_presentes,serie_completa
34,Norte,CSP da ULS Póvoa Varzim / Vila Conde,2024-01-01,2026-02-01,26,False
35,Norte,CSP da ULS Póvoa Varzim / Vila do Conde,2026-03-01,2026-06-01,4,False


Designações completas: 38


## 6. Preparar os totais regionais e as faixas de dimensão

As percentagens regionais são calculadas com **a soma das contagens a dividir pela soma dos inscritos**, no mesmo mês. Não são a média simples das percentagens das ULS e não somamos pessoas de meses diferentes.

Para a H4, definimos três faixas comuns de dimensão usando apenas o número de inscritos de janeiro de 2024. Os limites são os tercis dessa fotografia inicial e ficam fixos para todos os meses. Se uma unidade crescer e mudar de faixa, essa mudança ficará visível na tabela. A H4 é uma comparação dentro de cada mês; isto não transforma cada grupo numa coorte fixa.

Usar limites comuns permite comparar regiões na mesma faixa. Ainda assim, estar na mesma faixa não torna duas ULS iguais em dimensão. Grupos com menos de três unidades são assinalados; o limite de três é um aviso de fragilidade, não uma garantia estatística.

In [8]:
regional = (
    principal.groupby(["mes", "regiao_saude"])[CONTAGENS]
    .sum()
    .reset_index()
    .sort_values(["regiao_saude", "mes"])
)
for pct, contagem in RECALCULADAS.items():
    regional[pct] = 100 * regional[contagem] / regional["inscritos"]
regional = regional.merge(
    principal.groupby(["mes", "regiao_saude"])
    .size()
    .rename("unidades_no_mes")
    .reset_index(),
    on=["mes", "regiao_saude"],
    validate="one_to_one",
)

base_dimensao = principal.loc[principal["mes"].eq(INICIO), "inscritos"]
limite_1, limite_2 = [float(x) for x in base_dimensao.quantile([1 / 3, 2 / 3])]
if limite_1 >= limite_2:
    raise ValueError(
        "Não foi possível definir três faixas distintas. Rever a regra de dimensão."
    )
principal["faixa_dimensao"] = pd.cut(
    principal["inscritos"],
    bins=[float("-inf"), limite_1, limite_2, float("inf")],
    labels=["menor", "intermedia", "maior"],
    right=True,
)
series = principal.loc[principal["serie_completa"]].copy().reset_index(drop=True)
faixas = {
    "mes_referencia": INICIO.strftime("%Y-%m"),
    "variavel": "inscritos",
    "metodo": "tercis comuns do primeiro mês do recorte",
    "limite_1": limite_1,
    "limite_2": limite_2,
    "regra": "menor: inscritos <= limite_1; intermedia: limite_1 < inscritos <= limite_2; maior: inscritos > limite_2",
    "aviso_grupo_pequeno": "menos de 3 unidades; critério descritivo, sem garantia de suficiência estatística",
}
h4 = (
    principal.groupby(["mes", "faixa_dimensao", "regiao_saude"], observed=True)
    .agg(
        unidades=("unidade_fonte", "nunique"),
        inscritos=("inscritos", "sum"),
        sem_medico=("sem_medico", "sum"),
        mediana_sem_medico_pct=("sem_medico_pct", "median"),
        inscritos_min=("inscritos", "min"),
        inscritos_max=("inscritos", "max"),
    )
    .reset_index()
)
h4["sem_medico_pct_ponderada"] = 100 * h4["sem_medico"] / h4["inscritos"]
h4["grupo_pequeno"] = h4["unidades"].lt(3)

(PASTA_OUTPUT / "sns_faixas_dimensao.json").write_text(
    json.dumps(faixas, ensure_ascii=False, indent=2), encoding="utf-8"
)
guardar_csv(h4, "03_h4_regiao_dimensao_mensal.csv", PASTA_ANALYTICS)
print("Limites fixos de inscritos:", round(limite_1, 2), "e", round(limite_2, 2))
display(h4.loc[h4["mes"].eq(FIM)])
print("Estas tabelas preparam a H4; ainda não demonstram um efeito da região.")

Limites fixos de inscritos: 194226.33 e 327435.33


,mes,faixa_dimensao,regiao_saude,unidades,inscritos,sem_medico,mediana_sem_medico_pct,inscritos_min,inscritos_max,sem_medico_pct_ponderada,grupo_pequeno
319,2026-06-01,menor,Alentejo,4,508422,94908,19.448032,107563,167443,18.667170,False
320,2026-06-01,menor,Centro,4,441403,52254,12.827241,86351,140759,11.838161,False
321,2026-06-01,menor,Lisboa e Vale do Tejo,1,185232,62450,33.714477,185232,185232,33.714477,True
322,2026-06-01,menor,Norte,4,627054,9431,1.738624,126867,179250,1.504017,False
323,2026-06-01,intermedia,Centro,1,279778,6201,2.216400,279778,279778,2.216400,True
324,2026-06-01,intermedia,Lisboa e Vale do Tejo,6,1469033,557540,38.905397,206880,260064,37.952857,False
325,2026-06-01,intermedia,Norte,4,1057180,28448,1.321599,245532,291504,2.690932,False
326,2026-06-01,maior,Algarve,1,561074,122390,21.813522,561074,561074,21.813522,True
327,2026-06-01,maior,Centro,3,1184977,122492,4.941828,347847,419933,10.337078,False
328,2026-06-01,maior,Lisboa e Vale do Tejo,5,2276241,559174,25.676843,360466,582385,24.565676,False


Estas tabelas preparam a H4; ainda não demonstram um efeito da região.


## 7. Preparar o histórico e o alvo do mês seguinte

Cada linha desta tabela representa uma previsão feita com os dados do **mês de origem** para o **mês seguinte** da mesma unidade.

| Campo | Informação usada |
|---|---|
| `sem_medico_pct` | Percentagem do mês de origem. É também a previsão de referência: repetir o último valor. |
| `sem_medico_pct_lag1` | Percentagem um mês antes da origem. |
| `sem_medico_pct_lag3` | Percentagem três meses antes da origem. |
| `sem_medico_pct_media3` | Média da origem e dos dois meses anteriores. |
| `variacao_sem_medico_pp` | Diferença entre a origem e o mês anterior. |
| `variacao_inscritos_pct` | Variação dos inscritos entre a origem e o mês anterior. |
| `regiao_sem_medico_pct` | Percentagem regional no mês de origem, calculada com todas as unidades desse mês. |
| `alvo_sem_medico_pct` | Percentagem observada no mês seguinte. É a resposta a prever, nunca uma variável de entrada. |

A média inclui o mês de origem porque esse valor já é conhecido no momento definido para a previsão. O `shift(-1)` só serve para juntar a resposta futura usada no treino e na avaliação. Confirmamos que o mês do alvo é exatamente o seguinte, para não saltar lacunas.

Guardamos todos os pares consecutivos para a referência simples. Para os modelos com histórico, guardamos um subconjunto com as variáveis completas: os primeiros meses ainda não têm três meses anteriores. Não preenchemos esse histórico com zeros.

A fase `04` terá uma lista explícita de variáveis permitidas. `mes_alvo`, `alvo_sem_medico_pct` e a identificação do conjunto não podem entrar como preditores. As taxas de consultas ficam fora dessa lista enquanto a sua definição estiver pendente.

In [9]:
pares = series.sort_values(CHAVE).copy().reset_index(drop=True)
GRUPO = ["regiao_saude", "unidade_fonte"]
g = pares.groupby(GRUPO, sort=False)
pares["sem_medico_pct_lag1"] = g["sem_medico_pct"].shift(1)
pares["sem_medico_pct_lag3"] = g["sem_medico_pct"].shift(3)
pares["sem_medico_pct_media3"] = g["sem_medico_pct"].transform(
    lambda s: s.rolling(3, min_periods=3).mean()
)
pares["variacao_sem_medico_pp"] = pares["sem_medico_pct"] - pares["sem_medico_pct_lag1"]
pares["variacao_inscritos_pct"] = 100 * (
    pares["inscritos"] / g["inscritos"].shift(1) - 1
)
pares["mes_alvo"] = g["mes"].shift(-1)
pares["alvo_sem_medico_pct"] = g["sem_medico_pct"].shift(-1)
pares["previsao_ultimo_valor_pct"] = pares["sem_medico_pct"]

pares = pares.loc[pares["mes_alvo"].notna()].copy()
mes_seguinte = pares["mes"] + pd.offsets.MonthBegin(1)
if not pares["mes_alvo"].eq(mes_seguinte).all():
    raise ValueError(
        "Encontrado um par de meses não consecutivos. Rever a continuidade."
    )
pares = pares.merge(
    regional[["mes", "regiao_saude", "sem_medico_pct"]].rename(
        columns={"sem_medico_pct": "regiao_sem_medico_pct"}
    ),
    on=["mes", "regiao_saude"],
    validate="many_to_one",
)
pares["conjunto"] = "desenvolvimento"
pares.loc[pares["mes_alvo"].ge(INICIO_TESTE), "conjunto"] = "teste"

NUMERICAS_MODELO = [
    "inscritos",
    "sem_medico_pct",
    "sem_medico_opcao_pct",
    "sem_medico_pct_lag1",
    "sem_medico_pct_lag3",
    "sem_medico_pct_media3",
    "variacao_sem_medico_pp",
    "variacao_inscritos_pct",
    "regiao_sem_medico_pct",
]
CATEGORICAS_MODELO = ["regiao_saude"]
IDENTIFICACAO = ["regiao_saude", "unidade_fonte", "mes", "mes_alvo", "conjunto"]
COLUNAS_PARES = list(
    dict.fromkeys(
        IDENTIFICACAO
        + NUMERICAS_MODELO
        + ["alvo_sem_medico_pct", "previsao_ultimo_valor_pct"]
    )
)
pares = pares[COLUNAS_PARES].sort_values(GRUPO + ["mes"]).reset_index(drop=True)
modelo = pares.dropna(subset=NUMERICAS_MODELO).copy().reset_index(drop=True)
print("Pares consecutivos para a referência:", len(pares))
print("Pares com histórico completo para modelos:", len(modelo))
print("Pares sem histórico suficiente:", len(pares) - len(modelo))
display(
    modelo[
        [
            "unidade_fonte",
            "mes",
            "mes_alvo",
            "sem_medico_pct_lag3",
            "sem_medico_pct",
            "alvo_sem_medico_pct",
            "conjunto",
        ]
    ].head()
)

Pares consecutivos para a referência: 1102
Pares com histórico completo para modelos: 988
Pares sem histórico suficiente: 114


,unidade_fonte,mes,mes_alvo,sem_medico_pct_lag3,sem_medico_pct,alvo_sem_medico_pct,conjunto
0,CSP da ULS Alentejo Central,2024-04-01,2024-05-01,12.668503,11.835318,10.980543,desenvolvimento
1,CSP da ULS Alentejo Central,2024-05-01,2024-06-01,11.357310,10.980543,9.329947,desenvolvimento
2,CSP da ULS Alentejo Central,2024-06-01,2024-07-01,11.327425,9.329947,10.040928,desenvolvimento
3,CSP da ULS Alentejo Central,2024-07-01,2024-08-01,11.835318,10.040928,9.178075,desenvolvimento
4,CSP da ULS Alentejo Central,2024-08-01,2024-09-01,10.980543,9.178075,6.072926,desenvolvimento


## 8. Fixar a separação temporal e verificar o resultado

O teste final fica reservado para **os meses-alvo de janeiro a junho de 2026**. Uma linha com origem em dezembro de 2025 e alvo em janeiro de 2026 pertence ao teste, mesmo que a sua informação de entrada seja de 2025.

No `04`, o desenvolvimento pode usar uma validação temporal: treinar com meses-alvo até junho de 2025 e validar de julho a dezembro de 2025. Depois de escolher o modelo, treinar novamente com todo o desenvolvimento, até dezembro de 2025. O teste final não deve ser usado para escolher variáveis ou ajustar parâmetros.

Esta preparação não treina modelos, não calcula escalas de normalização e não ajusta codificações. Esses ajustes terão de usar só o treino, na fase seguinte. Em cada origem do teste, os meses anteriores já são conhecidos: podemos usá-los como histórico sem usar o alvo dessa linha.

Como não temos as versões antigas de cada publicação, o teste usa os valores da recolha atual. Pode incluir revisões que não eram conhecidas na altura. Vamos manter esta limitação na avaliação.

In [10]:
if (
    modelo.empty
    or not modelo["conjunto"].eq("teste").any()
    or not modelo["conjunto"].eq("desenvolvimento").any()
):
    raise ValueError("Não há dados suficientes para separar desenvolvimento e teste.")

checks = pd.DataFrame(
    [
        ["Chave mensal única no principal", not principal.duplicated(CHAVE).any()],
        [
            "30 meses e 39 registos por mês no principal",
            len(cobertura) == 30 and cobertura["linhas"].eq(39).all(),
        ],
        [
            "Séries completas com todos os meses",
            series.groupby(GRUPO)["mes"].nunique().eq(len(MESES)).all(),
        ],
        [
            "Alvo exatamente no mês seguinte",
            pares["mes_alvo"].eq(pares["mes"] + pd.offsets.MonthBegin(1)).all(),
        ],
        [
            "Variáveis do modelo e alvo sem valores em falta",
            not modelo[NUMERICAS_MODELO + CATEGORICAS_MODELO + ["alvo_sem_medico_pct"]]
            .isna()
            .any()
            .any(),
        ],
        [
            "Variáveis numéricas do modelo finitas",
            modelo[NUMERICAS_MODELO].abs().lt(float("inf")).all().all(),
        ],
        [
            "Desenvolvimento termina antes do teste",
            modelo.loc[modelo["conjunto"].eq("desenvolvimento"), "mes_alvo"].max()
            < modelo.loc[modelo["conjunto"].eq("teste"), "mes_alvo"].min(),
        ],
        [
            "Teste tem os 6 meses-alvo previstos",
            set(modelo.loc[modelo["conjunto"].eq("teste"), "mes_alvo"])
            == set(pd.date_range(INICIO_TESTE, FIM, freq="MS")),
        ],
        [
            "Referência e modelos têm as mesmas chaves no teste",
            set(map(tuple, pares.loc[pares["conjunto"].eq("teste"), CHAVE].to_numpy()))
            == set(
                map(tuple, modelo.loc[modelo["conjunto"].eq("teste"), CHAVE].to_numpy())
            ),
        ],
        [
            "Percentagens principais entre 0 e 100",
            principal[list(RECALCULADAS)].ge(0).all().all()
            and principal[list(RECALCULADAS)].le(100).all().all(),
        ],
    ],
    columns=["verificacao", "passou"],
)
checks["passou"] = checks["passou"].astype(bool)
guardar_csv(checks, "03_validacoes_finais.csv", PASTA_ANALYTICS)
display(checks)
if not checks["passou"].all():
    raise ValueError(
        "Há verificações finais que falharam. Rever antes de exportar os datasets."
    )

divisao = (
    modelo.groupby("conjunto")
    .agg(
        linhas=("unidade_fonte", "size"),
        unidades=("unidade_fonte", "nunique"),
        primeiro_mes_alvo=("mes_alvo", "min"),
        ultimo_mes_alvo=("mes_alvo", "max"),
    )
    .reset_index()
)
guardar_csv(divisao, "03_divisao_temporal.csv", PASTA_ANALYTICS)
display(divisao)

,verificacao,passou
0,Chave mensal única no principal,True
1,30 meses e 39 registos por mês no principal,True
2,Séries completas com todos os meses,True
3,Alvo exatamente no mês seguinte,True
4,Variáveis do modelo e alvo sem valores em falta,True
5,Variáveis numéricas do modelo finitas,True
6,Desenvolvimento termina antes do teste,True
7,Teste tem os 6 meses-alvo previstos,True
8,Referência e modelos têm as mesmas chaves no t...,True
9,Percentagens principais entre 0 e 100,True


,conjunto,linhas,unidades,primeiro_mes_alvo,ultimo_mes_alvo
0,desenvolvimento,760,38,2024-05-01,2025-12-01
1,teste,228,38,2026-01-01,2026-06-01


## 9. Guardar os dados preparados e o registo das decisões

| Ficheiro em `data/output/` | Uso |
|---|---|
| `sns_csp_principal.csv` | Todas as linhas do período principal; fotografias mensais, rankings e comparação geográfica. |
| `sns_csp_series_completas.csv` | Séries completas para comparar evolução entre janeiro de 2024 e junho de 2026. |
| `sns_csp_regional.csv` | Totais e percentagens ponderadas por região/mês, usando todas as linhas do principal. |
| `sns_csp_pares_mensais.csv` | Pares consecutivos para a referência simples, incluindo o início sem histórico suficiente para os modelos. |
| `sns_csp_modelacao.csv` | Pares com as variáveis históricas completas; desenvolvimento e teste identificados. |
| `sns_faixas_dimensao.json` | Limites fixos usados na comparação por dimensão. |
| `sns_preparacao_manifesto.json` | Fonte, contagens, assinaturas, variáveis permitidas e decisões desta execução. |

Os CSV preparados usam vírgula como separador e ponto decimal. Ao ler no `04`, usar `pd.read_csv(..., parse_dates=["mes", "mes_alvo"])` para a tabela de modelação. Para as outras tabelas, converter apenas `mes`. As categorias e os booleanos podem ser confirmados após leitura; os tipos originais ficam registados no dicionário.

**Pendente:** validar a correspondência territorial da Póvoa/Vila do Conde e a definição exata das taxas de consultas. As escolhas conservadoras permitem avançar com as restantes perguntas sem assumir uma confirmação que ainda não temos.

In [11]:
TABELAS = {
    "sns_csp_principal.csv": principal,
    "sns_csp_series_completas.csv": series,
    "sns_csp_regional.csv": regional,
    "sns_csp_pares_mensais.csv": pares,
    "sns_csp_modelacao.csv": modelo,
}
for nome, tabela in TABELAS.items():
    guardar_csv(tabela, nome)

dicionario_final = pd.DataFrame(
    [
        {"tabela": nome, "coluna": coluna, "tipo_em_memoria": str(tabela[coluna].dtype)}
        for nome, tabela in TABELAS.items()
        for coluna in tabela.columns
    ]
)
guardar_csv(dicionario_final, "03_tipos_tabelas_preparadas.csv", PASTA_ANALYTICS)
assinaturas_depois = {p.name: sha256(p) for p in ficheiros_input}
if assinaturas_antes != assinaturas_depois:
    raise ValueError("Os ficheiros de entrada mudaram durante a execução.")

manifesto = {
    "dataset": recolha["dataset"],
    "url_fonte": recolha["url_metadados"],
    "recolhido_em_utc": recolha["recolhido_em_utc"],
    "input_sha256": assinaturas_antes,
    "brutos_preservados": True,
    "inicio_principal": INICIO.strftime("%Y-%m-%d"),
    "fim_principal": FIM.strftime("%Y-%m-%d"),
    "linhas_historico": len(bruto),
    "linhas_fora_periodo": len(historico) - len(principal),
    "linhas_principal": len(principal),
    "designacoes_principal": int(principal["unidade_fonte"].nunique()),
    "linhas_series_completas": len(series),
    "designacoes_series_completas": int(series["unidade_fonte"].nunique()),
    "linhas_sem_continuidade_confirmada": len(principal) - len(series),
    "linhas_pares": len(pares),
    "linhas_modelacao": len(modelo),
    "linhas_pares_sem_historico_suficiente": len(pares) - len(modelo),
    "inicio_teste_mes_alvo": INICIO_TESTE.strftime("%Y-%m-%d"),
    "validacao_sugerida_mes_alvo": {
        "treino_ate": "2025-06-01",
        "validacao_de": "2025-07-01",
        "validacao_ate": "2025-12-01",
    },
    "variaveis_numericas_permitidas": NUMERICAS_MODELO,
    "variaveis_categoricas_permitidas": CATEGORICAS_MODELO,
    "alvo": "alvo_sem_medico_pct",
    "referencia": "previsao_ultimo_valor_pct",
    "alvo_unidade": "percentagem dos inscritos; erro avaliado em pontos percentuais",
    "transformacoes_ajustadas_em_todo_dataset": "nenhuma normalização ou codificação; faixas descritivas definidas no primeiro mês",
    "pendencias": [
        "Correspondência territorial das duas designações da Póvoa/Vila do Conde",
        "Janela exata das taxas de consultas de um ano",
    ],
    "limites": [
        "Continuidade do nome não garante continuidade territorial",
        "Recolha atual pode incluir revisões históricas",
        "Inscritos não são população residente",
        "Previsão histórica do indicador antigo; não extrapolar para julho de 2026",
    ],
    "outputs": [
        {"ficheiro": nome, "linhas": len(tabela), "sha256": sha256(PASTA_OUTPUT / nome)}
        for nome, tabela in TABELAS.items()
    ],
}
(PASTA_OUTPUT / "sns_preparacao_manifesto.json").write_text(
    json.dumps(manifesto, ensure_ascii=False, indent=2), encoding="utf-8"
)

nota = f"""# Fase 3 — Decisões de preparação

Recolha: {recolha["recolhido_em_utc"]} (UTC). Fonte: {recolha["url_metadados"]}.

- D01: {len(bruto)} linhas no histórico; {len(principal)} no recorte de 2024-01 a 2026-06; {len(historico) - len(principal)} fora do período. Brutos preservados.
- D02: nomes claros e tipos explícitos; unidade_fonte e periodo_fonte mantêm os valores da fonte. Chave: região, unidade e mês. Ver 03_mapa_colunas.csv.
- D03: qualidade verificada sem correções automáticas, imputação, interpolação ou remoção de duplicados. Diferença tolerada de arredondamento: 0,01 pp. Valores extremos válidos mantidos.
- D04: {principal["unidade_fonte"].nunique()} designações ao longo do recorte, 39 por mês. Não fundir os dois nomes da Póvoa/Vila do Conde sem confirmação territorial. {len(series)} linhas de {series["unidade_fonte"].nunique()} séries completas para evolução e previsão; {len(principal) - len(series)} linhas mantidas apenas no principal/regionais/H4.
- D05: percentagens recalculadas das contagens, publicadas preservadas. Região calculada com soma das contagens / soma dos inscritos, em cada mês.
- D06: limites das faixas comuns fixados pelos tercis dos inscritos de janeiro de 2024: {limite_1:.6f} e {limite_2:.6f}. Grupos com menos de três unidades assinalados; esta regra não garante suficiência estatística. A pertença à faixa pode mudar.
- D07: {len(pares)} pares consecutivos; {len(modelo)} com histórico completo. {len(pares) - len(modelo)} pares não usados nos modelos por falta de histórico inicial. Teste por mês-alvo, janeiro a junho de 2026. Validação sugerida: julho a dezembro de 2025, com treino anterior. Variáveis permitidas no manifesto; não ajustar modelos ou pré-processadores nesta fase.
- D08: taxas de consultas preservadas mas excluídas das variáveis do modelo e da interpretação da Q5 até confirmar a janela de um ano.

## Limitações e pendências

Os nomes estáveis não provam continuidade territorial. As coordenadas são pontos de referência. Inscritos não são a população residente. A recolha atual pode conter revisões. Não ligar a série antiga ao novo indicador de julho de 2026.

Continuam pendentes a correspondência territorial da Póvoa/Vila do Conde e a janela exata das consultas. A comparação temporal e a previsão cobrem 38 séries; rankings e totais regionais mantêm os 39 registos mensais.

A alteração ACES/ULS e a quebra de definição do indicador estão justificadas no 01 e verificadas no 02. Fontes: [API SNS]({recolha["url_metadados"]}) e [ERS — reorganização ULS](https://www.ers.pt/pt/atividade/supervisao/selecionar/alertas-de-supervisao/2024/alertas/alerta-de-supervis%C3%A3o-n-%C2%BA-1-2024/).

## Próxima fase

Usar sns_csp_modelacao.csv no 04. Comparar modelos com a referência simples nas mesmas linhas. Usar só as variáveis permitidas do manifesto. Ajustar codificações e escalas no treino, escolher o modelo pela validação temporal e reservar o teste final para avaliação.
"""
(PASTA_DOCS / "03_decisoes_preparacao.md").write_text(nota, encoding="utf-8")
display(pd.DataFrame(manifesto["outputs"])[["ficheiro", "linhas"]])
print("Preparação concluída. Ficheiros brutos preservados e verificações aprovadas.")

,ficheiro,linhas
0,sns_csp_principal.csv,1170
1,sns_csp_series_completas.csv,1140
2,sns_csp_regional.csv,150
3,sns_csp_pares_mensais.csv,1102
4,sns_csp_modelacao.csv,988


Preparação concluída. Ficheiros brutos preservados e verificações aprovadas.


## 10. O que fica pronto e o que falta

Ficam prontos o recorte principal, as séries completas, os totais regionais, a tabela mensal da H4 e os dados para comparar previsões a um mês. As verificações e as quantidades de linhas acompanham os ficheiros.

Na fase `04`, vamos comparar a referência de repetir o último valor com modelos simples. Na avaliação, os erros serão calculados nas mesmas unidades e meses. Para a análise descritiva, vamos usar o principal ou os totais regionais, conforme a pergunta — não o subconjunto da previsão por defeito.

As duas pendências de definição continuam visíveis. Se obtivermos confirmação que justifique alterar uma decisão, atualizamos a preparação e voltamos a executar as fases que dependem dela.

## Próxima fase

[Avançar para o 04 — Modelação](./04_modeling.ipynb)